In [1]:
!pip install bitsandbytes

In [2]:
import os
from huggingface_hub import HfFolder, login

# Sjekk om token allerede er lagret
if HfFolder.get_token() is None:
    login()
else:
    print("Allerede logget inn på Hugging Face 🤗")

Allerede logget inn på Hugging Face 🤗


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
import zipfile
import os

zip_path = "mistral-asl-instruct-working.zip"
extract_dir = "mistral-asl-instruct-working"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Mappen er pakket ut.")

Mappen er pakket ut.


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import torch

# Last PEFT-konfig
peft_config = PeftConfig.from_pretrained("./mistral-asl-instruct-working")

# 4-bit kvantisering
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Last basemodell
base_model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=quant_config
)

# Legg til LoRA
model = PeftModel.from_pretrained(base_model, "./mistral-asl-instruct-working")

# Last tokenizer
tokenizer = AutoTokenizer.from_pretrained("./mistral-asl-instruct-working", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Prompt-test
messages = [
    {"role": "user", "content": "Translate the following to American Sign Language (ASL) structure:\nWhat is your favourite movie?."}
]
prompt = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
output = model.generate(prompt, max_new_tokens=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] Translate the following to American Sign Language (ASL) structure:
What is your favourite movie?. [/INST] MOVIE, YOU FAVORITE WHAT? (what is your favorite movie?)


In [6]:
messages = [
    {"role": "user", "content": "Translate the following to American Sign Language (ASL) structure:\nDo you enjoy pizza or taco more?."}
]
prompt = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
output = model.generate(prompt, max_new_tokens=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] Translate the following to American Sign Language (ASL) structure:
Do you enjoy pizza or taco more?. [/INST] PIZZA YOU ENJOY MORE TACO YOU? (DO YOU ENJOY PIZZA MORE THAN TACOS?)
